# 02 — Test market-data (Redis cache outputs)

Smoke test du container `fxvol-market-data` — étape 2/5. Valide que **les 4 clés Redis produites par l'engine sont présentes, fraîches, numériques et se rafraîchissent en continu**. C'est le test fondamental du pipeline de cache que l'API REST consume pour servir les endpoints `/api/v1/spot/*`.

**Couvre** :
1. `heartbeat:market_data` présent + ISO-8601 parsable + age < 30s + TTL > 0
2. `latest_spot:EURUSD`, `latest_bid:EURUSD`, `latest_ask:EURUSD` présents avec valeurs numériques
3. Sanity prix : spot dans la fenêtre EUR/USD raisonnable [0.5, 2.0]
4. **Cohérence bid/ask/mid** : `bid ≤ mid ≤ ask` ET le spread `ask - bid` est positif et < 1% du mid (anti-quote-aberrante)
5. **Rafraîchissement** : sur 2 lectures espacées de 1s, au moins une des 3 clés (heartbeat ou spot/bid/ask) doit avoir changé — preuve que la boucle 100ms tourne et que IB pousse des ticks

**Préreq** :
- Notebook 01 passe vert (container healthy, env vars OK)
- Marché ouvert OU close récent dispo : si tu lances ce notebook un dimanche soir avec marché clos depuis vendredi, certaines valeurs (bid/ask) peuvent être absentes ou stale ; le close du vendredi sera dans `latest_spot` mais ne se rafraîchira plus → §5 rafraîchissement va FAIL légitimement
- Redis exposé sur l'host à `localhost:6380` (cf. `docker-compose.override.yml` en dev)

**Référence** : `src/services/market_data/engine.py` (loop 100ms, publish + cache), `src/bus/keys.py` (templates de keys), `src/bus/publisher.py` (set_heartbeat + publish_tick avec throttle 200ms).

## Setup — connexion Redis + helpers

Connexion via le port host `6380` exposé par `docker-compose.override.yml`. Pas de password (Redis ne tourne qu'en réseau Docker interne, pas exposé au LAN — défense en profondeur côté binding 127.0.0.1).

In [22]:
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"

# Fenêtre de prix raisonnable pour EUR/USD (historique : ~0.85-1.60).
# Marge confortable pour catcher tout sauf un prix manifestement faux.
PRICE_MIN = 0.5
PRICE_MAX = 2.0

# Spread max acceptable en % du mid. EUR/USD spot/futures = typiquement
# 0.5-2 pips sur 1.17 = ~0.001-0.002%. 1% = marge énorme, anti-quote-aberrante.
MAX_SPREAD_PCT = 1.0

results = []

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError(
        "Redis baseline KO. Container down ?\n"
        "  -> .\\scripts\\start_stack.ps1 ; vérifie 'docker compose ps' = redis healthy"
    )
print(f"Connected -> {REDIS_URL}")

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

Connected -> redis://localhost:6380/0


## 1. Heartbeat — présent, ISO-8601 valide, age < 30s, TTL > 0

**Ce que tu dois voir** : `heartbeat:market_data` retourne une string ISO-8601 (eg. `2026-04-28T14:15:16.368139Z`), parsable par `datetime.fromisoformat`, avec un age < 30s (push toutes les ~1s, donc au pire 1s vieille). TTL résiduel > 0 (TTL initial 30s, refreshé en continu).

**Si FAIL** :
- `heartbeat = None` → engine ne pousse pas (vérifier logs `docker logs fxvol-market-data`)
- `age > 30s` → engine est freezé / ne tourne plus dans la boucle
- `TTL = -1` ou `-2` → bug : `set` sans `ex=30` ou clé déjà expirée et pas réécrite
- `ValueError` au parse → format autre que ISO-8601 (ne devrait pas arriver après le fix compose)

In [23]:
print("== 1. heartbeat ==")

hb_raw = r.get("heartbeat:market_data")
record("heartbeat:market_data présent", hb_raw is not None,
       hb_raw if hb_raw else "<missing>")

if hb_raw:
    try:
        hb_dt = datetime.fromisoformat(hb_raw.replace("Z", "+00:00"))
        age_s = (datetime.now(timezone.utc) - hb_dt).total_seconds()
        record("heartbeat ISO-8601 parsable", True, f"parsed = {hb_dt.isoformat()}")
        record("heartbeat age < 30s", age_s < 30, f"age = {age_s:.2f}s")
    except ValueError as e:
        record("heartbeat ISO-8601 parsable", False, f"raw={hb_raw!r}: {e}")

    ttl = r.ttl("heartbeat:market_data")
    record("heartbeat TTL > 0", ttl > 0, f"TTL = {ttl}s")

== 1. heartbeat ==
  [OK  ] heartbeat:market_data présent  | 2026-04-29T09:04:10.576946Z
  [OK  ] heartbeat ISO-8601 parsable  | parsed = 2026-04-29T09:04:10.576946+00:00
  [OK  ] heartbeat age < 30s  | age = 0.74s
  [OK  ] heartbeat TTL > 0  | TTL = 29s


## 2. Cache prix — `latest_spot/bid/ask:EURUSD` présents et numériques

**Ce que tu dois voir** : les 3 clés présentes, valeurs castables en `float`, TTL > 0. C'est le cache que l'`api` REST utilise pour servir `/api/v1/spot/EURUSD`.

**Si FAIL** :
- `latest_spot` absent mais heartbeat OK → bug dans la logique `_publish_ticks_to_redis` (engine reçoit des heartbeats mais pas de ticks IB ?)
- `latest_bid/ask` absents mais `latest_spot` OK → bug similaire ou throttle qui drop trop
- valeur non castable en float → corruption (rare, signaler à l'engine)

In [24]:
print("== 2. cache prix ==")

prices = {}
for kind in ("spot", "bid", "ask"):
    key = f"latest_{kind}:{SYMBOL}"
    raw = r.get(key)
    if raw is None:
        record(f"{key} présent", False, "<missing>")
        continue
    try:
        val = float(raw)
        prices[kind] = val
        ttl = r.ttl(key)
        record(f"{key} numérique + TTL", True,
               f"{val:.5f} (TTL {ttl}s)")
    except (TypeError, ValueError):
        record(f"{key} numérique", False, f"raw={raw!r} non-castable en float")

== 2. cache prix ==
  [OK  ] latest_spot:EURUSD numérique + TTL  | 1.17025 (TTL 30s)
  [OK  ] latest_bid:EURUSD numérique + TTL  | 1.17024 (TTL 30s)
  [OK  ] latest_ask:EURUSD numérique + TTL  | 1.17026 (TTL 30s)


## 3. Sanity prix — spot dans la fenêtre EUR/USD raisonnable

**Ce que tu dois voir** : `spot ∈ [0.5, 2.0]`. EUR/USD historique : ~0.85-1.60 sur les 20 dernières années. Notre fenêtre catche tout sauf une valeur manifestement fausse (0, négatif, en cents au lieu de devise, autre paire renvoyée par erreur, etc.).

In [25]:
print("== 3. sanity prix ==")

spot = prices.get("spot")
if spot is None:
    record("sanity prix", False, "latest_spot absent (cf. §2)")
else:
    in_range = PRICE_MIN < spot < PRICE_MAX
    record(f"spot dans [{PRICE_MIN}, {PRICE_MAX}]", in_range,
           f"{spot:.5f}" + ("" if in_range else " (HORS FENÊTRE)"))

== 3. sanity prix ==
  [OK  ] spot dans [0.5, 2.0]  | 1.17025


## 4. Cohérence bid/ask/mid + spread raisonnable

**Invariants attendus** :
- `bid ≤ mid ≤ ask` (le mid est entre les deux côtés du book)
- `ask > bid` (spread positif strict ; un spread = 0 ou négatif = quote inversée = aberration)
- `(ask - bid) / mid < 1%` (anti-quote-aberrante : un spread > 1% sur EUR/USD = bug ou marché ultra-volatile, à investiguer)

**Note marché fermé** : weekend ou off-hours, `bid` et `ask` peuvent être absents (uniquement `close` populé côté tick IB → engine ne calcule pas de mid). Dans ce cas on skip ce check avec un message explicatif.

In [26]:
print("== 4. cohérence bid/ask/mid ==")

bid, ask, spot = prices.get("bid"), prices.get("ask"), prices.get("spot")

if bid is None or ask is None or spot is None:
    record("cohérence bid/ask/mid", False,
           f"bid={bid}, ask={ask}, spot={spot} — au moins un absent (marché fermé ?)")
else:
    record("bid ≤ spot ≤ ask",
           bid <= spot <= ask,
           f"{bid:.5f} ≤ {spot:.5f} ≤ {ask:.5f}")
    record("spread positif (ask > bid)",
           ask > bid,
           f"ask - bid = {(ask - bid)*10000:.2f} pips")
    spread_pct = ((ask - bid) / spot) * 100
    record(f"spread < {MAX_SPREAD_PCT}% du mid",
           spread_pct < MAX_SPREAD_PCT,
           f"spread = {spread_pct:.4f}%")

== 4. cohérence bid/ask/mid ==
  [OK  ] bid ≤ spot ≤ ask  | 1.17024 ≤ 1.17025 ≤ 1.17026
  [OK  ] spread positif (ask > bid)  | ask - bid = 0.20 pips
  [OK  ] spread < 1.0% du mid  | spread = 0.0017%


## 5. Rafraîchissement — la boucle 100ms tourne et IB pousse des ticks

**Le test** : 2 snapshots du heartbeat + des 3 prix, espacés de 1s. Au moins **une des 4 valeurs** doit avoir changé entre les 2 snapshots. Sinon → la boucle est freezée OU IB ne pousse plus de ticks (déconnexion silencieuse, marché fermé, etc.).

**Note marché fermé** : si bid/ask/spot sont les mêmes mais le **heartbeat a changé**, c'est OK — l'engine tourne, simplement IB ne pousse pas de tick frais (close du vendredi). Le test passe tant que **au moins une valeur** bouge.

In [27]:
print("== 5. rafraîchissement (2 snapshots à 1s) ==")

def snapshot():
    return {
        "heartbeat": r.get("heartbeat:market_data"),
        "spot": r.get(f"latest_spot:{SYMBOL}"),
        "bid": r.get(f"latest_bid:{SYMBOL}"),
        "ask": r.get(f"latest_ask:{SYMBOL}"),
    }

snap1 = snapshot()
time.sleep(1.0)
snap2 = snapshot()

changed = [k for k in snap1 if snap1[k] != snap2[k]]
record("≥ 1 valeur a changé en 1s",
       len(changed) > 0,
       f"changed: {changed}" if changed else "snapshots identiques (engine freezé ou marché fermé sans push heartbeat ?)")

# Cas particulier informatif : si seule la heartbeat change, marché probablement fermé
if changed == ["heartbeat"]:
    print("  [INFO] seul le heartbeat bouge — marché probablement fermé, ticks IB stables")

== 5. rafraîchissement (2 snapshots à 1s) ==
  [OK  ] ≥ 1 valeur a changé en 1s  | changed: ['heartbeat', 'spot', 'bid']


## Récap final

In [28]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — Redis cache outputs sains. Pass au notebook 03 (pub/sub channel ticks).")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
heartbeat:market_data présent                           OK      2026-04-29T09:04:10.576946Z
heartbeat ISO-8601 parsable                             OK      parsed = 2026-04-29T09:04:10.576946+00:00
heartbeat age < 30s                                     OK      age = 0.74s
heartbeat TTL > 0                                       OK      TTL = 29s
latest_spot:EURUSD numérique + TTL                      OK      1.17025 (TTL 30s)
latest_bid:EURUSD numérique + TTL                       OK      1.17024 (TTL 30s)
latest_ask:EURUSD numérique + TTL                       OK      1.17026 (TTL 30s)
spot dans [0.5, 2.0]                                    OK      1.17025
bid ≤ spot ≤ ask                                        OK      1.17024 ≤ 1.17025 ≤ 1.17026
spread positif (ask > bid)                              OK 

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| `heartbeat absent` | engine crashé ou Redis pas joignable depuis market-data | `docker logs fxvol-market-data --tail 30` ; relance `docker compose --profile engines up -d --force-recreate market-data` |
| `heartbeat age > 30s` | engine freezé (loop bloquée), ou IB connecté mais ticks ne flowent pas | `docker logs fxvol-market-data` ; vérifier ib-gateway healthy ; restart |
| `ValueError au parse heartbeat` | format de timestamp inattendu (ne devrait pas arriver post-fix compose) | Vérifier `src/bus/publisher.py:set_heartbeat` |
| `latest_spot absent` mais heartbeat OK | engine reçoit pas de ticks IB (TrustedIPs ?) | Voir `00 SMOKE_PLAN.md` § 4 ; vérifier que l'IP de market-data est dans Trusted IPs côté ib-gateway |
| `latest_bid/ask absent` mais spot OK | marché fermé (off-hours) ou throttle qui drop | Comportement normal weekend ; sinon investiguer `src/bus/publisher.py:publish_tick` |
| `spot HORS FENÊTRE` | quote aberrante côté IB ou symbole erroné | Vérifier `MARKET_SYMBOL` env var (= `EURUSD`) ; logs market-data |
| `bid > spot > ask` (cohérence FAIL) | bug calcul de mid côté engine | Investiguer `src/services/market_data/engine.py` |
| `spread > 1%` | quote temporairement large (fin de session, news) ou bug | Acceptable si transitoire ; si persistant → vérifier ticker IB |
| `aucune valeur ne bouge en 1s` | engine freezé OU marché fermé sans push heartbeat | Si seul heartbeat bouge → marché fermé (OK). Si rien ne bouge → restart container |